In [1]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable

!pip install selenium
!pip install webdriver-manager
!pip install beautifulsoup4

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://dl.google.com/linux/chrome/deb stable InRelease [2,548 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [359 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,410 B]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,002 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:13 http://archive.ubuntu.com/ubuntu noble-b

In [7]:
# ============================================================
# 日経新聞のWebサイトから日経平均株価を取得する
# ============================================================

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup

from datetime import datetime
from zoneinfo import ZoneInfo

from urllib.parse import (
    urlsplit,
    urlunsplit,
    parse_qsl,
    urlencode
)

import re
import time


JST = ZoneInfo("Asia/Tokyo")


# ============================================================
# URLのパラメータを変更する関数
# ============================================================

def change_query_parameter(url, key, value):

    parts = urlsplit(url)

    query = dict(
        parse_qsl(
            parts.query,
            keep_blank_values=True
        )
    )

    query[key] = value

    new_query = urlencode(query)

    return urlunsplit(
        (
            parts.scheme,
            parts.netloc,
            parts.path,
            new_query,
            parts.fragment
        )
    )


# ============================================================
# 株価の表示を整える関数
# ============================================================

def format_price(value):

    if value is None:
        return None

    value = value.replace(",", "")

    try:

        if "." in value:

            decimal_length = len(
                value.split(".")[1]
            )

            number = float(value)

            return f"{number:,.{decimal_length}f}"

        number = int(value)

        return f"{number:,}"

    except ValueError:

        return value


# ============================================================
# 日付を YYYY-MM-DD に変換する関数
# ============================================================

def parse_date_text(text):

    # 2026/09/16
    # 2026-09-16
    # 2026.09.16

    match = re.search(
        r"(?<!\d)"
        r"(20\d{2})"
        r"\s*[/\-.]\s*"
        r"(\d{1,2})"
        r"\s*[/\-.]\s*"
        r"(\d{1,2})"
        r"(?!\d)",
        text
    )

    if match:

        year = int(match.group(1))
        month = int(match.group(2))
        day = int(match.group(3))

        if 1 <= month <= 12 and 1 <= day <= 31:

            return (
                f"{year:04d}-"
                f"{month:02d}-"
                f"{day:02d}"
            )


    # 2026年9月16日

    match = re.search(
        r"(?<!\d)"
        r"(20\d{2})年"
        r"\s*"
        r"(\d{1,2})月"
        r"\s*"
        r"(\d{1,2})日",
        text
    )

    if match:

        year = int(match.group(1))
        month = int(match.group(2))
        day = int(match.group(3))

        if 1 <= month <= 12 and 1 <= day <= 31:

            return (
                f"{year:04d}-"
                f"{month:02d}-"
                f"{day:02d}"
            )


    # 9月16日

    match = re.search(
        r"(?<!\d)"
        r"(\d{1,2})月"
        r"\s*"
        r"(\d{1,2})日",
        text
    )

    if match:

        month = int(match.group(1))
        day = int(match.group(2))

        if 1 <= month <= 12 and 1 <= day <= 31:

            today = datetime.now(JST)

            year = today.year

            if month > today.month:
                year -= 1

            return (
                f"{year:04d}-"
                f"{month:02d}-"
                f"{day:02d}"
            )


    # 9/16

    match = re.search(
        r"(?<![\d.])"
        r"(\d{1,2})"
        r"\s*/\s*"
        r"(\d{1,2})"
        r"(?![\d.])",
        text
    )

    if match:

        month = int(match.group(1))
        day = int(match.group(2))

        if 1 <= month <= 12 and 1 <= day <= 31:

            today = datetime.now(JST)

            year = today.year

            if month > today.month:
                year -= 1

            return (
                f"{year:04d}-"
                f"{month:02d}-"
                f"{day:02d}"
            )

    return None


# ============================================================
# Step2
# HTMLから株価データを抽出する関数
# ============================================================

def extract_stock_data(html):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # 始値、高値、安値を含む領域を探す

    candidates = []

    for tag in soup.find_all(
        [
            "div",
            "section",
            "ul",
            "li",
            "table",
            "tbody",
            "tr",
            "g"
        ]
    ):

        text = tag.get_text(
            " ",
            strip=True
        )

        if not text:
            continue

        if (
            "始値" in text
            and "高値" in text
            and "安値" in text
        ):

            if len(text) <= 1500:

                candidates.append(
                    (
                        len(text),
                        tag,
                        text
                    )
                )


    # 最も小さい領域を使用

    if candidates:

        candidates.sort(
            key=lambda x: x[0]
        )

        data_tag = candidates[0][1]
        data_text = candidates[0][2]

    else:

        data_tag = None

        data_text = " ".join(
            soup.stripped_strings
        )


    # 株価を取得する関数

    def find_price(label):

        pattern = (
            re.escape(label)
            + r"\s*"
            + r"[:：]?\s*"
            + r"("
            + r"[0-9][0-9,]*"
            + r"(?:\.[0-9]+)?"
            + r")"
            + r"(?![\d,])"
        )

        match = re.search(
            pattern,
            data_text
        )

        if match:

            return format_price(
                match.group(1)
            )

        return None


    # 始値、高値、安値、終値

    open_price = find_price("始値")

    high_price = find_price("高値")

    low_price = find_price("安値")

    close_price = find_price("終値")


    # 終値ではなく現在値と表示される場合

    if close_price is None:

        close_price = find_price(
            "現在値"
        )


    # 日付を取得

    date = parse_date_text(
        data_text
    )


    # データ領域の親要素から日付を探す

    if (
        date is None
        and data_tag is not None
    ):

        parent = data_tag.parent

        count = 0

        while (
            parent is not None
            and count < 5
        ):

            parent_text = parent.get_text(
                " ",
                strip=True
            )

            date = parse_date_text(
                parent_text
            )

            if date is not None:
                break

            parent = parent.parent

            count += 1


    # tooltip、date、timeなどから日付を探す

    if date is None:

        date_candidates = []

        for tag in soup.find_all(True):

            classes = " ".join(
                tag.get(
                    "class",
                    []
                )
            )

            element_id = tag.get(
                "id",
                ""
            )

            attributes = (
                classes
                + " "
                + element_id
            ).lower()


            if any(
                word in attributes
                for word in [
                    "tooltip",
                    "date",
                    "time",
                    "label"
                ]
            ):

                text = tag.get_text(
                    " ",
                    strip=True
                )

                if (
                    text
                    and len(text) <= 200
                ):

                    parsed = parse_date_text(
                        text
                    )

                    if parsed:

                        date_candidates.append(
                            (
                                len(text),
                                parsed
                            )
                        )


        if date_candidates:

            date_candidates.sort(
                key=lambda x: x[0]
            )

            date = date_candidates[0][1]


    # 必要な値が揃っていなければNone

    if date is None:
        return None

    if open_price is None:
        return None

    if high_price is None:
        return None

    if low_price is None:
        return None

    if close_price is None:
        return None


    # 日付、始値、高値、安値、終値をリストで返す

    return [
        date,
        open_price,
        high_price,
        low_price,
        close_price
    ]


# ============================================================
# 日経ページ内のチャートを探す関数
# ============================================================

def find_chart_container(driver):

    elements = driver.find_elements(
        By.CSS_SELECTOR,
        '[class*="Chart_container"]'
    )

    candidates = []

    for element in elements:

        try:

            if not element.is_displayed():
                continue

            width = element.size["width"]
            height = element.size["height"]

            if (
                width >= 400
                and height >= 200
            ):

                candidates.append(
                    (
                        width * height,
                        element
                    )
                )

        except Exception:

            continue


    if not candidates:
        return None


    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )


    return candidates[0][1]


# ============================================================
# Highchartsのグラフを探す関数
# ============================================================

def find_graph(driver):

    elements = driver.find_elements(
        By.CSS_SELECTOR,
        "svg.highcharts-root"
    )

    candidates = []

    for element in elements:

        try:

            if not element.is_displayed():
                continue

            width = element.size["width"]
            height = element.size["height"]

            if (
                width >= 400
                and height >= 150
            ):

                candidates.append(
                    (
                        width * height,
                        element
                    )
                )

        except Exception:

            continue


    if not candidates:
        return None


    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )


    return candidates[0][1]


# ============================================================
# Step3
# Webページから株価データを抽出する関数
# ============================================================

def get_stock_values(
    driver,
    url
):

    # 日経ページへアクセス

    driver.get(url)


    wait = WebDriverWait(
        driver,
        30
    )


    wait.until(
        EC.presence_of_element_located(
            (
                By.TAG_NAME,
                "body"
            )
        )
    )


    # チャートを読み込むためスクロール

    for _ in range(5):

        driver.execute_script(
            """
            window.scrollBy(
                0,
                window.innerHeight * 0.6
            );
            """
        )

        time.sleep(0.5)


    # チャートを取得

    chart_container = (
        find_chart_container(
            driver
        )
    )


    if chart_container is None:

        raise Exception(
            "チャートを見つけられませんでした。"
        )


    # iframeを取得

    iframes = chart_container.find_elements(
        By.TAG_NAME,
        "iframe"
    )


    if not iframes:

        raise Exception(
            "iframeを見つけられませんでした。"
        )


    iframe = iframes[0]


    iframe_src = iframe.get_attribute(
        "src"
    )


    # 6か月表示に変更

    six_month_url = (
        change_query_parameter(
            iframe_src,
            "auto_interval",
            "6m"
        )
    )


    # 6か月チャートへアクセス

    driver.get(
        six_month_url
    )


    WebDriverWait(
        driver,
        30
    ).until(
        EC.presence_of_element_located(
            (
                By.TAG_NAME,
                "body"
            )
        )
    )


    time.sleep(2)


    # Highchartsのグラフを取得

    graph = find_graph(
        driver
    )


    if graph is None:

        raise Exception(
            "グラフを見つけられませんでした。"
        )


    graph_width = int(
        graph.size["width"]
    )


    # グラフを中央へ表示

    driver.execute_script(
        """
        arguments[0].scrollIntoView(
            {
                block: "center",
                inline: "center"
            }
        );
        """,
        graph
    )


    time.sleep(0.5)


    stock_values = {}


    # グラフ右端・左端の座標

    right_edge = (
        graph_width // 2
        - 5
    )

    left_edge = (
        -graph_width // 2
        + 5
    )


    # グラフ右端から左端まで1pxずつ移動

    for x_offset in range(
        right_edge,
        left_edge - 1,
        -1
    ):

        try:

            ActionChains(
                driver,
                duration=1
            ).move_to_element_with_offset(
                graph,
                x_offset,
                0
            ).perform()


            time.sleep(0.008)


            # 現在のHTMLを取得

            html = driver.page_source


            # Beautiful Soupで株価データを抽出

            stock_data = (
                extract_stock_data(
                    html
                )
            )


            if stock_data is not None:

                date = stock_data[0]

                stock_values[
                    date
                ] = stock_data


        except Exception:

            break


    # 日付順に並べる

    result = []

    for date in sorted(
        stock_values.keys()
    ):

        result.append(
            stock_values[
                date
            ]
        )


    return result


# ============================================================
# Step4
# main
# ============================================================

url = (
    "https://www.nikkei.com/"
    "markets/worldidx/chart/"
    "nk225/?type=6month"
)


# Chromeの設定

chrome_options = (
    webdriver.ChromeOptions()
)

chrome_options.add_argument(
    "--headless=new"
)

chrome_options.add_argument(
    "--no-sandbox"
)

chrome_options.add_argument(
    "--disable-dev-shm-usage"
)

chrome_options.add_argument(
    "--disable-gpu"
)

chrome_options.add_argument(
    "--window-size=1920,1080"
)


# ChromeDriverを起動

service = Service(
    ChromeDriverManager().install()
)


driver = webdriver.Chrome(
    service=service,
    options=chrome_options
)


try:

    # スクレイピング開始時間

    start_time = datetime.now(
        JST
    )


    print(
        "スクレイピング開始:",
        start_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )


    # 株価データを取得

    stock_values = (
        get_stock_values(
            driver,
            url
        )
    )


    # スクレイピング終了時間

    end_time = datetime.now(
        JST
    )


    print(
        "スクレイピング終了:",
        end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )


    # スクレイピングにかかった時間

    elapsed_time = (
        end_time
        - start_time
    )


    print(
        "スクレイピングにかかった時間:",
        elapsed_time
    )


    print()


    # 日付、始値、高値、安値、終値を表示

    for stock_data in stock_values:

        print(
            stock_data[0],
            stock_data[1],
            stock_data[2],
            stock_data[3],
            stock_data[4]
        )


finally:

    driver.quit()

スクレイピング開始: 2026-09-16 18:19:46
スクレイピング終了: 2026-09-16 18:22:05
スクレイピングにかかった時間: 0:02:19.576654

2026-04-08 54,386.65 56,424.63 54,380.02 56,308.42
2026-04-09 56,199.86 56,406.49 55,763.05 55,895.32
2026-04-10 56,265.77 57,012.77 56,251.18 56,924.11
2026-04-13 56,421.46 56,765.72 56,232.78 56,502.77
2026-04-14 57,085.65 57,979.82 57,010.18 57,877.39
2026-04-15 58,265.18 58,585.95 58,028.75 58,134.24
2026-04-16 58,479.83 59,688.1 58,428.19 59,518.34
2026-04-17 59,255.09 59,381.25 58,475.9 58,475.9
2026-04-20 58,821.16 59,169.13 58,687.96 58,824.89
2026-04-21 59,031.51 59,611.91 59,004.76 59,349.17
2026-04-22 59,104.11 59,708.21 59,005.48 59,585.86
2026-04-23 59,758.64 60,013.98 58,621.48 59,140.23
2026-04-24 59,407.44 59,763.68 59,225.37 59,716.18
2026-04-27 59,880.71 60,903.95 59,608.63 60,537.36
2026-04-28 60,531.78 60,634.66 59,701.84 59,917.46
2026-04-30 59,484.71 59,560.57 58,928.2 59,284.92
2026-05-01 59,379.12 59,706.7 59,263.5 59,513.12
2026-05-07 60,241.31 63,091.14 60,213.02 62,8